In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Classical feedforward และ control flow (dynamic circuits)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# Classical feedforward และ control flow


<details>
<summary><b>เวอร์ชันของแพ็กเกจ</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ข้อกำหนดดังต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Dynamic circuits เวอร์ชันใหม่พร้อมให้บริการแก่ผู้ใช้ทุกคนบน Backend ทุกตัวแล้ว ตอนนี้สามารถรัน dynamic circuits ในระดับ utility scale ได้ ดู [ประกาศ](/announcements/product-updates/2025-09-25-new-dynamic-circuits) สำหรับรายละเอียดเพิ่มเติม

Dynamic circuits เป็นเครื่องมือที่ทรงพลัง ซึ่งช่วยให้วัดค่า Qubit ได้ในระหว่างการรัน quantum circuit และจากนั้นดำเนินการ classical logic ภายใน Circuit โดยอิงจากผลของการวัดกลางวงจร (mid-circuit measurements) กระบวนการนี้เรียกอีกอย่างว่า _classical feedforward_ แม้ยังเป็นช่วงต้นของการเข้าใจวิธีใช้ประโยชน์จาก dynamic circuits ได้ดีที่สุด แต่ชุมชนวิจัยควอนตัมก็ระบุกรณีการใช้งานหลายอย่างไว้แล้ว เช่น:

* การเตรียม quantum state อย่างมีประสิทธิภาพ เช่น [GHZ state,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [W-state,](https://arxiv.org/abs/2403.07604) (สำหรับข้อมูลเพิ่มเติมเกี่ยวกับ W-state ดูได้ที่ ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) และ [matrix product states](https://arxiv.org/abs/2404.16083) หลากหลายประเภท
* [การพัวพันระยะไกลอย่างมีประสิทธิภาพ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) ระหว่าง Qubit บนชิปเดียวกันโดยใช้ shallow circuits
* [การสุ่มตัวอย่าง IQP-like circuits](https://arxiv.org/pdf/2505.04705) อย่างมีประสิทธิภาพ

อย่างไรก็ตาม การปรับปรุงที่ dynamic circuits นำมานั้นมีข้อแลกเปลี่ยน การวัดกลางวงจรและ classical operations โดยทั่วไปใช้เวลารันนานกว่า two-qubit gates และเวลาที่เพิ่มขึ้นนี้อาจทำให้ประโยชน์ของการลด circuit depth หมดไป ดังนั้น การลดระยะเวลาการวัดกลางวงจรจึงเป็นจุดโฟกัสของการปรับปรุงขณะที่ IBM Quantum&reg; เปิดตัว [เวอร์ชันใหม่](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) ของ dynamic circuits

[ข้อกำหนด OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) กำหนดโครงสร้าง control-flow หลายแบบ แต่ Qiskit Runtime รองรับเฉพาะคำสั่งเงื่อนไข `if` เท่านั้น ใน Qiskit SDK จะสอดคล้องกับเมธอด [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) บน [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) เมธอดนี้คืนค่า [context manager](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) และมักใช้ในคำสั่ง `with` คู่มือนี้อธิบายวิธีใช้คำสั่งเงื่อนไขดังกล่าว

> **Note:** ตัวอย่างโค้ดในคู่มือนี้ใช้คำสั่ง measure มาตรฐานสำหรับ mid-circuit measurements อย่างไรก็ตาม แนะนำให้ใช้คำสั่ง [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit) แทน หาก Backend รองรับ ดูรายละเอียดที่ [เอกสาร Mid-circuit measurements](/guides/measure-qubits#mid-circuit-measurements)
## คำสั่ง `if`
คำสั่ง `if` ใช้เพื่อดำเนินการตามเงื่อนไขโดยอิงจากค่าของ classical bit หรือ register

ในตัวอย่างด้านล่าง เราจะใส่ Hadamard Gate ให้กับ Qubit แล้ววัดค่า ถ้าผลลัพธ์เป็น 1 เราจะใส่ X Gate บน Qubit ซึ่งมีผลพลิกกลับไปที่สถานะ 0 จากนั้นเราวัด Qubit อีกครั้ง ผลการวัดสุดท้ายควรเป็น 0 ด้วยความน่าจะเป็น 100%

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
# Use MidCircuitMeasure() if it's supported by the backend.
# circuit.append(MidCircuitMeasure(), [q0], [c0])
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

คำสั่ง `with` สามารถกำหนด assignment target ซึ่งเป็น context manager ที่สามารถเก็บไว้และนำไปใช้สร้าง else block ได้ โดย else block จะถูกรันเมื่อเนื้อหาใน `if` block *ไม่ถูก*รัน

ในตัวอย่างด้านล่าง เราจะกำหนด register ที่มี Qubit สองตัวและ classical bit สองบิต เราใส่ Hadamard Gate ให้กับ Qubit ตัวแรกแล้ววัดค่า ถ้าผลลัพธ์เป็น 1 เราจะใส่ Hadamard Gate บน Qubit ตัวที่สอง มิเช่นนั้น เราจะใส่ X Gate บน Qubit ตัวที่สอง สุดท้ายเราวัด Qubit ตัวที่สองด้วย

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

นอกจากการกำหนดเงื่อนไขบน classical bit เดียวแล้ว ยังสามารถกำหนดเงื่อนไขบนค่าของ classical register ที่ประกอบด้วยหลายบิตได้

ในตัวอย่างด้านล่าง เราใส่ Hadamard Gate ให้กับ Qubit สองตัวแล้ววัดค่า ถ้าผลลัพธ์เป็น `01` หมายถึง Qubit ตัวแรกเป็น 1 และ Qubit ตัวที่สองเป็น 0 เราจะใส่ X Gate ให้กับ Qubit ตัวที่สาม สุดท้ายเราวัด Qubit ตัวที่สาม โปรดสังเกตว่าเพื่อความชัดเจน เราเลือกระบุสถานะของ classical bit ตัวที่สาม ซึ่งเป็น 0 ไว้ในเงื่อนไข `if` ด้วย ในการวาด Circuit เงื่อนไขแสดงด้วยวงกลมบน classical bit ที่ถูกกำหนดเงื่อนไข วงกลมสีดำหมายถึงเงื่อนไขบน 1 ส่วนวงกลมสีขาวหมายถึงเงื่อนไขบน 0

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Classical expressions
โมดูล classical expression ของ Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) ประกอบด้วยการแสดงผลเชิงทดลองของการดำเนินการ runtime บนค่า classical ระหว่างการรัน Circuit เนื่องจากข้อจำกัดของฮาร์ดแวร์ ปัจจุบันรองรับเฉพาะเงื่อนไขของ `QuantumCircuit.if_test()` เท่านั้น

ตัวอย่างต่อไปนี้แสดงว่าสามารถใช้การคำนวณ parity เพื่อสร้าง n-qubit GHZ state โดยใช้ dynamic circuits ได้ อันดับแรก สร้าง $n/2$ Bell pairs บน Qubit ที่อยู่ติดกัน จากนั้น เชื่อมต่อคู่เหล่านี้โดยใช้ layer ของ CNOT gates ระหว่างคู่ จากนั้นวัด target Qubit ของ CNOT gates ทั้งหมดก่อนหน้าและ reset แต่ละ Qubit ที่วัดแล้วให้กลับเป็นสถานะ $\vert 0 \rangle$ ใส่ $X$ ให้กับทุก site ที่ไม่ถูกวัดซึ่ง parity ของบิตก่อนหน้าทั้งหมดเป็นเลขคี่ สุดท้าย ใส่ CNOT gates ให้กับ Qubit ที่ถูกวัดเพื่อสร้าง entanglement ขึ้นใหม่ที่สูญเสียไปจากการวัด

ในการคำนวณ parity องค์ประกอบแรกของ expression ที่สร้างขึ้นเกี่ยวข้องกับการยก Python object `mr[0]` ขึ้นเป็น node [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` ใช้เพื่อแปลง arbitrary objects ให้เป็น classical expressions) ไม่จำเป็นสำหรับ `mr[1]` และ classical register ที่ตามมา เนื่องจากเป็น inputs ให้กับ `expr.bit_xor` และการยกที่จำเป็นจะเกิดขึ้นโดยอัตโนมัติในกรณีเหล่านี้ expressions ดังกล่าวสามารถสร้างขึ้นใน loops และโครงสร้างอื่นได้

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## ค้นหา Backend ที่รองรับ dynamic circuits
หากต้องการค้นหา Backend ทั้งหมดที่บัญชีของคุณเข้าถึงได้และรองรับ dynamic circuits ให้รันโค้ดดังต่อไปนี้ ตัวอย่างนี้สมมติว่าคุณ [บันทึก login credentials ไว้แล้ว](/guides/save-credentials) นอกจากนี้ยังสามารถ [ระบุ credentials อย่างชัดเจน](/guides/initialize-account#explicit) เมื่อเริ่มต้น Qiskit Runtime service account ซึ่งช่วยให้ดู Backend ที่มีให้บริการบน instance หรือประเภทแผนที่เฉพาะเจาะจงได้

> **Note:** - Backend ที่บัญชีเข้าถึงได้ขึ้นอยู่กับ instance ที่ระบุใน credentials
> - Dynamic circuits เวอร์ชันใหม่พร้อมให้บริการแก่ผู้ใช้ทุกคนบน Backend ทุกตัวแล้ว ดู [ประกาศ](/announcements/product-updates/2025-09-25-new-dynamic-circuits) สำหรับรายละเอียดเพิ่มเติม

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_boston')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_miami')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_torino')>, <IBMBackend('ibm_kingston')>]


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use Sampler and build your own measurement circuits instead. Alternatively, you can use the [Executor primitive,](/docs/guides/directed-execution-model#executor-primitive) which supports dynamic circuits.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API,](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting) for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility  [`measurement_bases` module](https://github.com/Qiskit/qiskit-addon-utils/blob/38ea05431f2e9830adf4ec9265f6d19758a32096/qiskit_addon_utils/exp_vals/measurement_bases.py) for more information. [Get started with utilities.](/docs/guides/qiskit-addons-utils#get-started-with-utilities)
3. Add back together the results for each partition.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch.](/docs/guides/stretch)
- Learn about the shorter [mid-circuit measurements](/docs/guides/measure-qubits#mid-circuit-measurements) that reduce the circuit time.
- Use [circuit schedule visualization](/docs/guides/visualize-circuit-timing#qiskit-runtime-support) to debug and optimize your dynamic circuits.
</Admonition>